# Fundações da Previsão de Demandas de TI

O modelo tradicional de suporte (NTI) costuma ser reativo, gerando picos inesperados. O foco primário deste cenário é prever o volume de chamados de forma proativa. Faremos isdo usando dados reais gerados no ServiceNow, aplicando o método estatístico de análise e aprendizado supervisionado de _Time Series Forecasting_ através das camadas refinadas (Medallion Architecture).

In [ ]:
"""
==============================================================================
1. IMPORTAÇÕES E CONFIGURAÇÕES INICIAIS
==============================================================================
"""

import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

import sys
sys.path.append('..')
from src.data_processing import TemporalFeaturesExtractor, DataCleaner, TimeSeriesAggregator, load_bronze_data
from src.model import ModelEvaluator


## 1. Origem: Ingestão da Camada Bronze

Iniciamos capturando a massa de dados bruta (ITSM Management Dataset). A camada Bronze preserva o formato natural logo após a ingestão original do sistema ServiceNow.

In [ ]:
bronze_path = "../data/bronze/ITSM_data.csv"
if os.path.exists(bronze_path):
    df_bronze = load_bronze_data(bronze_path)
    print(f"Dimensão Dataframe Bronze: {df_bronze.shape}")
    display(df_bronze.head(3))
else:
    print("Arquivo base não detectado; prossiga gerando configuração mock.")


## 2. Abstração de Regras: A Camada de Configuração

Evitando o antipadrão de valores inseridos diretamente via _hardcoding_, extraímos parâmetros de negócio para esquemas de inicialização YAML (padrão DRY). Essas configurações gerem o tratamento semântico adiante.

In [ ]:
"""
==============================================================================
3. GERAÇÃO DE CONFIGURAÇÕES DE NEGÓCIO
==============================================================================
--- 3.1. Criação do Arquivo de Schema ---
"""

yaml_content = """\
business_rules:
  priority_imputation_matrix:
    "1 - High_1 - High": 1
    "1 - High_2 - Medium": 2
    "1 - High_3 - Low": 3
    "2 - Medium_1 - High": 2
    "2 - Medium_2 - Medium": 3
    "2 - Medium_3 - Low": 4
    "3 - Low_1 - High": 3
    "3 - Low_2 - Medium": 4
    "3 - Low_3 - Low": 5
  time_settings:
    expected_format: "%d/%m/%Y %H:%M"
  rare_labels_threshold: 0.05
"""

os.makedirs("../config", exist_ok=True)
with open("../config/schema.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)
print(
    "[Info] Variáveis mockadas estabelecidas no diretório de suporte (/config/schema.yaml)."
)


## 3. Tratamento e Transformação: O Componente Temporal

Em problemas preditivos, o fluxo cronológico contém o maior potencial de correlações matemáticas. Identificar comportamentos cíclicos convertendo-os em funções trigonométricas resolve falhas da numeração estática de meses e preserva seu fluxo periódico ao modelo de IA.

In [ ]:
"""
Classes movidas para src.data_processing. TemporalFeaturesExtractor importada com sucesso.
"""


## 4. Normalização: Motor de Regras e Limpeza (Data Cleaner)

Construímos a segunda perna dos filtros: classes dedicadas à higienização semântica. Suas missões variam desde reduzir o ruído aglutinando categorias menos comuns e eliminando sujeiras textuais do volume financeiro das transações de tempo de atendimento.

In [ ]:
"""
Classes movidas para src.data_processing. DataCleaner importada com sucesso.
"""


## 5. Aplicação do Pipeline: Geração de Camada Silver

Encapsulando todo o fluxo de saneamento (Single Source of Quality), injetamos os dados Bronze dentro da esteira de limpeza projetada nas classes acima.

In [ ]:
"""
==============================================================================
6. EXECUÇÃO INTEGRADA: DE BRONZE PARA SILVER
==============================================================================
--- 6.1. Transformação via Orientação ao Objeto ---
"""

if "df_bronze" in locals() and not df_bronze.empty:
    with open("../config/schema.yaml", "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    time_cols = ["Open_Time", "Resolved_Time", "Close_Time"]
    expected_format = config["business_rules"]["time_settings"]["expected_format"]
    temp_extractor = TemporalFeaturesExtractor(
        time_columns=time_cols, expected_format=expected_format
    )

    df_silver = temp_extractor.standardize_datetimes(df_bronze)
    df_silver = temp_extractor.add_cyclical_features(df_silver, target_col="Open_Time")

    df_silver["Flag_Reabertura"] = df_silver["Reopen_Time"].notnull().astype(int)

    cleaner = DataCleaner("../config/schema.yaml")
    cols_to_drop = [
        "WBS",
        "KB_number",
        "Reopen_Time",
        "No_of_Related_Incidents",
        "No_of_Related_Changes",
        "Related_Change",
    ]

    df_silver = cleaner.drop_noise_columns(df_silver, cols_to_drop)
    df_silver = cleaner.impute_priority_with_matrix(df_silver)
    df_silver = cleaner.clean_handle_time(df_silver, "Handle_Time_hrs")

    cat_cols = [
        "Category",
        "CI_Cat",
        "CI_Subcat",
        "Closure_Code",
        "Status",
        "Alert_Status",
    ]
    df_silver = cleaner.group_rare_categories(df_silver, categorical_columns=cat_cols)

    print(f"Shape Final da Silver Layer: {df_silver.shape}")
    display(df_silver.head(2))


## 6. A Camada Gold: O Tempo Contínuo

Modelos multivariados e previsões dependentes do tempo não apreciam lacunas. Na Camada Gold adaptamos a série histórica consolidando falhas diárias e colapsando frequências para que todo os registros de chamados operacionais unifiquem num somador volumétrico (Agregador).

In [ ]:
"""
Classes movidas para src.data_processing. TimeSeriesAggregator importada com sucesso.
"""


In [ ]:
"""--- 7.2. Validação da Camada Contínua ---"""

if "df_silver" in locals():
    aggregator = TimeSeriesAggregator(time_index_col="Open_Time", freq="D")
    df_gold_global = aggregator.generate_global_pulse(df_silver)

    print(f"Estrutura Sequencial Definitiva (Gold): {df_gold_global.shape}")
    display(df_gold_global.head())


## 7. Deep Dive Series: O Som e A Autocorrelação

Através da avaliação da `Decomposição Aditiva` notamos os impactos severos gerados aos finais de semana e sazonalidades ruidosas em janelas de 7 dias.

In [ ]:
"""
==============================================================================
8. DECOMPOSIÇÃO E AUTOCORRELAÇÃO
==============================================================================
--- 8.1. Compreendendo O Racional Intríseco da Série ---
"""

if "df_gold_global" in locals():
    plt.rcParams["figure.figsize"] = (14, 8)
    decomposicao = sm.tsa.seasonal_decompose(
        df_gold_global["Volume_Global"], model="additive", period=7
    )
    fig = decomposicao.plot()
    plt.suptitle("Decomposição: Tendência, Sazonalidade (7D) e Residual", y=1.02)
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    plot_acf(
        df_gold_global["Volume_Global"],
        lags=30,
        ax=ax[0],
        title="Função de Autocorrelação (ACF)",
    )
    plot_pacf(
        df_gold_global["Volume_Global"],
        lags=30,
        ax=ax[1],
        title="Autocorrelação Parcial (PACF)",
    )
    plt.show()


## 8. Abordagem Metódica de Validação (MLOps)

Simulação preditiva requer cautela para impedir vazamento de dados futuros (`data leakage`). Para aferirmos qualquer avanço na modelagem inteligente, criamos uma regressão de baseline clássica empregando a Média Móvel de Sete Dias (`SMA-7`) deslocada estrategicamente num corte de Cross Validation estritamente temporal (`TimeSeriesSplit`).

In [ ]:
"""
==============================================================================
9. VALIDAÇÃO TIME-SERIES E MODELO CHAMPION (XGBOOST)
==============================================================================
--- 9.1. Análise MLOps Multimodelo ---
"""

if "df_gold_global" in locals() and not df_gold_global.empty:
    evaluator = ModelEvaluator(n_splits=5, target_col="Volume_Global")
    
    # 1. BASELINE
    print("TREINANDO BASELINE SMA-7...")
    res_baseline = evaluator.evaluate_baseline(df_gold_global, window=7)
    print(f"Baseline -> Avaliação: RMSE = {res_baseline['mean_rmse']:.2f} | MAE = {res_baseline['mean_mae']:.2f}")
    
    # 2. XGBOOST
    print("\nTREINANDO XGBOOST CHAMPION...")
    # Recuperando as features temporais criadas na Silver
    features_cols = ["Workload_Global"] # Simplificando uso das targets como feature baseada e gerando shift
    # Para o xgboost vamos construir features de lags no momento antes de treinar
    df_eval = df_gold_global.copy()
    for lag in [1, 2, 7]:
        df_eval[f"Volume_lag_{lag}"] = df_eval["Volume_Global"].shift(lag)
    df_eval.dropna(inplace=True)
    
    res_xgb = evaluator.evaluate_xgboost(df_eval, features=[f"Volume_lag_{l}" for l in [1, 2, 7]] + ["Workload_Global"])
    print(f"XGBoost  -> Avaliação: RMSE = {res_xgb['mean_rmse']:.2f} | MAE = {res_xgb['mean_mae']:.2f}")
    
    y_truth = res_xgb['truth']
    y_forecast_sma = res_baseline['predictions'].loc[y_truth.index]
    y_forecast_xgb = res_xgb['predictions'].loc[y_truth.index]


## 9. Dashboard de Resíduos e Simulação da Frente Operacional

Fechamos a jornada analisando quão severos estão os erros finais. O gráfico avalia distâncias residuais garantindo a mensuração da margem em verde (previsão acima) contra vermelho (surpresa de carga e chamados estourando o teto).

In [ ]:
"""
==============================================================================
10. VISUALIZAÇÃO PREDITIVA DASHBOARD COMPARATIVA
==============================================================================
"""

def plot_forecast_comparative(y_true, y_base, y_xgb, dates):
    plt.figure(figsize=(16, 6))
    plt.plot(dates, y_true, label="Verdadeiro (Real)", color="black", linewidth=2)
    plt.plot(dates, y_base, label="Baseline (SMA-7)", color="orange", linestyle="--")
    plt.plot(dates, y_xgb, label="XGBoost (Champion)", color="green", linewidth=2)
    plt.title("Comparativo Preditivo (MLOps)")
    plt.legend()
    plt.show()

if "y_truth" in locals() and "y_forecast_sma" in locals() and "y_forecast_xgb" in locals():
    plot_forecast_comparative(y_truth, y_forecast_sma, y_forecast_xgb, y_truth.index)
